In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"neerajmittal099","key":"17b6b79614a0bab818cf141aeceae0e3"}'}

In [2]:
import os
data_dir = r"C:\path\to\chest_xray\chest_xray"

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
 27% 642M/2.29G [00:43<01:44, 17.1MB/s]

In [ ]:
import zipfile
zip_ref=zipfile.ZipFile('/content/chest-xray-pneumonia.zip')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D,Flatten, Dense, Rescaling, Dropout

In [ ]:
train_full_ds=keras.utils.image_dataset_from_directory(
    directory='/content/chest_xray/chest_xray/train',
    labels='inferred',
    label_mode='int',
    batch_size=None,
    image_size=(256, 256),
    shuffle=True,
    seed=42
)
test_ds=keras.utils.image_dataset_from_directory(
    directory='/content/chest_xray/chest_xray/test',
    labels='inferred',
    label_mode='int',
    batch_size=None,
    image_size=(256, 256),
    shuffle=False
)
total_train=train_full_ds.cardinality().numpy()
train_size=int(total_train * 0.85)
val_size=total_train - train_size

train_ds=train_full_ds.take(train_size)
val_ds=train_full_ds.skip(train_size)

print(f"Train:{train_ds.cardinality().numpy()}")
print(f"Val:{val_ds.cardinality().numpy()}")
print(f"Test:{test_ds.cardinality().numpy()}")

In [ ]:
# total_size=full_ds.cardinality().numpy()

In [ ]:
# total_size

In [ ]:
# train_size=int(total_size*0.7)
# test_size=int(total_size*0.15)
# val_size=int(total_size*0.15)

In [ ]:
# train_ds=full_ds.take(train_size)
# remain=full_ds.skip(train_size)
# val_ds=remain.take(val_size)
# test_ds=remain.skip(val_size)

In [ ]:
data_aug=tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2)
])

In [ ]:
autotune=tf.data.AUTOTUNE
batch_size=32

@tf.function
def preprocess_and_augment(x,y):
    x=tf.cast(x,tf.float32)/255.0
    x=data_aug(x,training=True)
    return x,y

@tf.function
def preprocess_only(x,y):
    x=tf.cast(x,tf.float32)/255.0
    return x,y
train_df=train_ds.map(preprocess_and_augment,num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)
val_df=val_ds.map(preprocess_only,num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)
test_df=test_ds.map(preprocess_only,num_parallel_calls=autotune).batch(batch_size).prefetch(autotune)
for x,y in train_df.take(1):
    print("Image min/max:", tf.reduce_min(x).numpy(), tf.reduce_max(x).numpy())

In [ ]:
val_ds.cardinality().numpy()

In [ ]:
class_names=['NORMAL','PNEUMONIA']

In [ ]:
import matplotlib.pyplot as plt
images,labels=next(iter(train_df))
plt.figure(figsize=(10,15))
for i in range(9):
  plt.subplot(3,3,i+1)
  plt.imshow(images[i].numpy())
  plt.title(class_names[labels[i].numpy()])

In [ ]:
# model=Sequential([
#     Input(shape=(256, 256, 3)),
#     Conv2D(32, (3,3), activation='relu', padding='same'),
#     MaxPooling2D(),
#     Conv2D(64, (3,3), activation='relu', padding='same'),
#     MaxPooling2D(),
#     Conv2D(128, (3,3), activation='relu', padding='same'),
#     MaxPooling2D(),
#     Conv2D(256, (3,3), activation='relu', padding='same'),
#     MaxPooling2D(),
#     tf.keras.layers.GlobalAveragePooling2D(),
#     Dense(256, activation='relu'),
#     Dropout(0.5),
#     Dense(1, activation='sigmoid')
# ])

In [ ]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
#     loss='binary_crossentropy',
#     metrics=[
#         'accuracy',
#         tf.keras.metrics.AUC(name='auc'),
#         tf.keras.metrics.Precision(name='precision'),
#         tf.keras.metrics.Recall(name='recall'),
#         tf.keras.metrics.TruePositives(name='tp'),
#         tf.keras.metrics.TrueNegatives(name='tn'),
#         tf.keras.metrics.FalsePositives(name='fp'),
#         tf.keras.metrics.FalseNegatives(name='fn'),
#     ]
# )

In [ ]:
# model.build(input_shape=(None,256,256,3))

In [ ]:
# model.summary()

In [ ]:
base_model=tf.keras.applications.MobileNetV2(
    input_shape=(256,256,3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable=False

model=Sequential([
    Input(shape=(256,256,3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, clipnorm=1.0),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.TruePositives(name='tp'),
        tf.keras.metrics.TrueNegatives(name='tn'),
        tf.keras.metrics.FalsePositives(name='fp'),
        tf.keras.metrics.FalseNegatives(name='fn'),
    ]
)

In [ ]:
model.build(input_shape=(None,256,256,3))

In [ ]:
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        mode='min'
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'best_model.keras',
        monitor='val_recall',
        save_best_only=True,
        mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    tf.keras.callbacks.CSVLogger(
        'training_log.csv',
        append=False
    )
]

In [ ]:
model.fit(
    train_df,
    epochs=63,
    verbose=1,
    validation_data=val_df,
    callbacks=callbacks,
    class_weight={0: 1.9546, 1: 0.6719}
)

In [ ]:
model.save('my_model_3.keras')